# ECON 5140 – Homework 3, Problem 5
## Propensity Score Model: AI Assistant Feature Rollout

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# ── Data Generation (same DGP as the problem set) ────────────────────────────
np.random.seed(123)
n = 600
plan_tier      = np.random.choice([0, 1, 2], n, p=[0.5, 0.35, 0.15])
tenure_days    = np.random.randint(0, 366, n)
prior_sessions = np.random.poisson(12, n) + np.random.randint(0, 20, n)
platform       = np.random.choice([0, 1], n, p=[0.6, 0.4])
signup_cohort  = np.random.choice([0, 1, 2], n, p=[0.4, 0.35, 0.25])

# Treatment: higher plan_tier / tenure / prior_sessions / mobile → more likely to be treated
logit_p = (-2.0 + 0.5*plan_tier + 0.003*tenure_days
            + 0.02*prior_sessions + 0.4*platform + 0.2*signup_cohort)
ps_true = 1 / (1 + np.exp(-np.clip(logit_p, -10, 10)))
D = (np.random.uniform(0, 1, n) < ps_true).astype(int)

# Outcome: true causal effect of feature = +15 min/week
Y = (80 + 15*D + 8*plan_tier + 0.05*tenure_days
     + 0.8*prior_sessions + 5*platform + 3*signup_cohort
     + np.random.normal(0, 12, n))
Y = np.maximum(Y, 0)

df = pd.DataFrame({
    'plan_tier': plan_tier, 'tenure_days': tenure_days,
    'prior_sessions': prior_sessions, 'platform': platform,
    'signup_cohort': signup_cohort, 'D': D, 'weekly_active_minutes': Y
})

print(f"n = {n},  Treated: {D.sum()},  Control: {n - D.sum()}")
print(f"True causal effect: +15 min/week")
df.head()

## Step 1 – Estimate Propensity Score via Logistic Regression

In [ ]:
covariates = ['plan_tier', 'tenure_days', 'prior_sessions', 'platform', 'signup_cohort']
X = df[covariates].values

lr = LogisticRegression(max_iter=1000)
lr.fit(X, D)

# Print model summary
print("Logistic Regression – Coefficients:")
print("-" * 40)
for name, coef in zip(covariates, lr.coef_[0]):
    print(f"  {name:20s}: {coef:+.4f}")
print(f"  {'Intercept':20s}: {lr.intercept_[0]:+.4f}")
print()
print("Interpretation:")
print("  Largest positive coefficients → strongest drivers of treatment.")
print("  prior_sessions and plan_tier both have large positive coefficients,")
print("  meaning heavier users on higher-tier plans are most likely to get access.")

## Step 2 – Add Fitted Propensity Scores to Dataset

In [ ]:
# Pr(D=1 | X) for each unit
df['ps'] = lr.predict_proba(X)[:, 1]

print("Propensity score summary by group:")
print(df.groupby('D')['ps'].describe().round(4))

## Step 3 – Check Overlap (Common Support)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

df[df['D'] == 0]['ps'].plot.kde(ax=ax, label='Control (D=0)', color='steelblue', linewidth=2)
df[df['D'] == 1]['ps'].plot.kde(ax=ax, label='Treated (D=1)',  color='coral',     linewidth=2)

ax.set_xlabel('Estimated Propensity Score  e(X)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Propensity Score Overlap: Treated vs Control', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Overlap assessment
ps_treated = df[df['D']==1]['ps']
ps_control = df[df['D']==0]['ps']
overlap_min = max(ps_treated.min(), ps_control.min())
overlap_max = min(ps_treated.max(), ps_control.max())
print(f"\nCommon support region: [{overlap_min:.3f}, {overlap_max:.3f}]")
n_in_support = ((df['ps'] >= overlap_min) & (df['ps'] <= overlap_max)).sum()
print(f"Units within common support: {n_in_support}/{len(df)} ({100*n_in_support/len(df):.1f}%)")
print("\nConclusion: Overlap is sufficient for IPW estimation.")
print("  The two distributions share substantial common support.")
print("  Control units with high e(X) exist to match treated units near the top.")

## Step 4 – Compute ATE via IPW and Compare

In [ ]:
# Clip propensity scores to avoid extreme weights (standard practice)
ps_clip = np.clip(df['ps'], 0.01, 0.99)

# Horvitz-Thompson IPW estimator for ATE:
#   ATE_IPW = (1/n) * Σ [ D_i * Y_i / e(X_i)  −  (1−D_i) * Y_i / (1−e(X_i)) ]
ipw_terms = np.where(D == 1,
                      Y / ps_clip,
                     -Y / (1 - ps_clip))
ATE_ipw = ipw_terms.mean()

# Naive (unadjusted) ATE
naive_ATE = (df[df['D'] == 1]['weekly_active_minutes'].mean()
           - df[df['D'] == 0]['weekly_active_minutes'].mean())

TRUE_EFFECT = 15.0

print("=" * 48)
print(f"  Naive ATE (unadjusted)  : {naive_ATE:+.2f} min/week")
print(f"  IPW ATE                 : {ATE_ipw:+.2f} min/week")
print(f"  True causal effect      : {TRUE_EFFECT:+.2f} min/week")
print("=" * 48)
print(f"  Naive bias              : {naive_ATE - TRUE_EFFECT:+.2f} min")
print(f"  IPW  bias               : {ATE_ipw  - TRUE_EFFECT:+.2f} min")
print()
print("Interpretation:")
print(f"  The naive estimate ({naive_ATE:.1f}) overstates the true effect ({TRUE_EFFECT:.0f})")
print(f"  by {naive_ATE-TRUE_EFFECT:.1f} min because treated users are power users who would")
print(f"  be more active regardless of the feature (positive selection bias).")
print(f"  IPW reweights the sample to balance observed covariates, reducing")
print(f"  the bias to {ATE_ipw-TRUE_EFFECT:.1f} min and recovering an estimate close to 15.")